# Hands-On AI for Science
## August 14, Afternoon: Computer Vision for Scientists

### Self-contained Colab notebook

Everything this notebook needs is either in it or downloaded by its own cells. Just run the cells in order. No GPU is required.


This morning, we introduced foundation models, and showed how representations can be extracted from a protein language model and probed for biological information.  This session applies the *same procedure* to a different kind of scientific data: **images**.  Microscopy, histology, gel photos, camera traps, satellite tiles — images are among the most common data types in science, and the workflow practiced this morning transfers to them almost unchanged.

The example task: classifying single blood cells by type from microscope images — something clinical laboratories literally do by eye, thousands of times a day.

## Setup

The cell below checks packages and files.  **If anything prints MISSING, fix it now**: the packages are installed with `pip install torch transformers torchvision` in a terminal, followed by a kernel restart (Kernel → Restart Kernel).  A MISSING file means Jupyter was started outside the repository's `lectures` folder.

The vision model is checked in Section 3, and — like this morning — the notebook has a built-in fallback if it can't be loaded.

### Data files

Run this cell to download the data files this notebook uses. You do not need to edit it.

In [ ]:
# The data files this notebook uses, fetched from the workshop repository.
# Run this cell once; you do not need to edit it.
import os, urllib.request

_FILES = [
    "blood_cells.npz",
    "image_embeddings.npy",
]

for _name in _FILES:
    if not os.path.exists(_name):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/jhasegaw/hands_on_ai_for_science/main/lectures/' + _name, _name)
print('Ready:', ', '.join(_FILES))

### Your code goes here: `a14pm_hw1alt`

The functions below are the ones you need to write. **Edit them right here in this cell**, then re-run this cell (Shift+Enter) to update your definitions. Re-run the cells further down to test them.

In [ ]:
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def embed_images(model, processor, images, batch_size=32):
    '''
    Extract one embedding per image from a pretrained vision model.

    For each batch of images (process in chunks of batch_size):
      1. Preprocess:  batch = processor(images=list_of_images,
         return_tensors='pt')
      2. Forward pass (inside torch.no_grad()):
         hidden = model(**batch).last_hidden_state    # (b, 1+patches, d)
      3. Keep the CLS token -- position 0 -- for each image:
         hidden[:, 0].  (The CLS token is the model's built-in summary of
         the whole image; this is the alternative to the mean-pooling used
         for proteins this morning.)

    @param:
    model - a HuggingFace AutoModel (frozen; never trained here)
    processor - the matching AutoImageProcessor
    images (array or list, each image (H, W, 3) uint8)
    batch_size (int): images per forward pass

    @return:
    embeddings (ndarray of float32, shape (len(images), hidden_size))

    Check: your output should reproduce the shipped image_embeddings.npy
    (generated with exactly this recipe -- see make_image_embeddings.py).
    '''
    raise RuntimeError("You need to write this part!")


def color_histogram(images, bins=8):
    '''
    The deliberately simple alternative representation: each image becomes
    its color distribution.

    For each image, histogram each of the 3 color channels into `bins`
    equal-width bins over the range [0, 256), concatenate the three
    histograms, and divide by the total count -- so each row is a set of
    3*bins fractions summing to 1.

    @param:
    images (array, shape (n, H, W, 3), uint8)
    bins (int): bins per channel

    @return:
    X (ndarray, shape (n, 3*bins)): rows of fractions summing to 1

    Hint: np.histogram(im[:, :, c], bins=bins, range=(0, 256))[0]
    '''
    raise RuntimeError("You need to write this part!")


def probe(X, y, groups=None):
    '''
    Measure how much information about y is linearly decodable from a
    frozen representation X, using cross-validated accuracy.

    This is yesterday's probe with ONE upgrade: the features are
    standardized INSIDE each fold, using a Pipeline --
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)) --
    so that scaling statistics are always computed from the training rows
    only (the leakage rule from Thursday morning).  cross_val_score
    handles the rest.

    If groups is given, use GroupKFold(5) with those groups; otherwise
    KFold(5, shuffle=True, random_state=0).  Return the mean accuracy.

    @param:
    X (array, shape (n, d)): representations
    y (array, shape (n,)): labels
    groups (array or None): group ids for grouped splitting

    @return:
    acc (float): mean cross-validated accuracy
    '''
    raise RuntimeError("You need to write this part!")

In [ ]:
import importlib.util, os

for pkg in ['torch', 'transformers', 'torchvision', 'numpy', 'matplotlib', 'sklearn', 'scipy']:
    print(f'{pkg:24s}', 'OK' if importlib.util.find_spec(pkg) is not None else 'MISSING')
for f in ['blood_cells.npz', 'image_embeddings.npy']:
    print(f'{f:24s}', 'OK' if os.path.exists(f) else 'MISSING')


1. [Images as data](#images)
1. [Convolution, by demonstration](#convolution)
1. [Extracting representations](#extracting)
1. [The comparison, third time](#comparison)
1. [Looking at the space](#looking)
1. [Homework](#homework)
1. [The recipe, vision edition](#recipe)

<a id="images"></a>

## 1. Images as data

An image is just an array.  A 64×64 color image is an array of shape (64, 64, 3): height, width, and three color channels (red, green, blue), each entry an integer from 0 to 255.  Wednesday's framing applies with no modification. Every dataset becomes an (n × features) matrix. This time, features just naturally happen to arrange themselves in a literal grid, a fact which our models can take advantage of.

The dataset: 600 microscope images of single blood cells, 75 from each of 8 cell types (from BloodMNIST — Yang et al. 2023, *Scientific Data* 10:41, [doi:10.1038/s41597-022-01721-8](https://doi.org/10.1038/s41597-022-01721-8) — built on Acevedo et al. 2020, *Data in Brief* 30:105474, [doi:10.1016/j.dib.2020.105474](https://doi.org/10.1016/j.dib.2020.105474); CC BY 4.0; see `make_blood_dataset.py` for provenance):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.load('blood_cells.npz')
images, labels = z['images'], z['labels']
class_names = ['basophil', 'eosinophil', 'erythroblast', 'immature granulocyte',
               'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

print('images:', images.shape, images.dtype, '   labels:', labels.shape)
print('one corner of one image (red channel):')
print(images[0, :4, :4, 0])

In [ ]:
fig = plt.figure(figsize=(14, 4), layout='tight')
axs = fig.subplots(2, 4)
for c, ax in enumerate(axs.flat):
    example = np.where(labels == c)[0][0]
    ax.imshow(images[example])
    ax.set_title(class_names[c], fontsize=12)
    ax.axis('off')

Telling these apart is a real skill — hematologists train for years to do it.  The question for this hour: how much of that skill is available, for free, from a model that has never seen a blood cell label?

<a id="convolution"></a>

## 2. Convolution, by demonstration

Before reaching for a big pretrained model, we will spend a few minutes on the idea that powered computer vision for a decade: **convolution**.  A small grid of numbers — a **kernel** — slides across the image. At each position, the overlapping pixels are multiplied by the kernel and summed.  Different kernels extract different things.  Watching three of them work tells most of the story:

In [ ]:
from scipy.signal import convolve2d

cell_gray = images[np.where(labels == 6)[0][0]].mean(axis=2)   # a neutrophil, grayscale

kernels = {
    'blur (5x5 average)': np.ones((5, 5)) / 25,
    'vertical edges': np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]]),
    'horizontal edges': np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]]),
}

fig = plt.figure(figsize=(14, 3.5), layout='tight')
axs = fig.subplots(1, 4)
axs[0].imshow(cell_gray, cmap='gray'); axs[0].set_title('original'); axs[0].axis('off')
for ax, (name, k) in zip(axs[1:], kernels.items()):
    ax.imshow(convolve2d(cell_gray, k, mode='valid'), cmap='gray')
    ax.set_title(name, fontsize=11); ax.axis('off')

Each kernel is a tiny pattern detector. The blur kernel responds to overall brightness, the edge kernels light up wherever intensity changes in their direction.  The mathematics of this operation — and the subtle difference between convolution and correlation — is developed carefully, with implementations from scratch, in `a14pm_cnn.ipynb` in this repository, which this section is indebted to.

The connection to modern vision models:

* A **convolutional neural network (CNN)** is this idea, learned and stacked. Instead of three hand-designed kernels (blur, vertical edge, horizontal edge), we use *thousands of learned ones*. These learned kernels are arranged in layers. Early layers detect edges, middle layers combine edges into textures and parts, and late layers combine parts into objects.  Nobody designs the kernels; they emerge from training.
* The newest vision models — including the one used next — are **vision transformers**. They cut the image into a grid of patches and process the patches with attention, the same architecture we used with *text* this morning.  An image becomes a "sentence" of patch-tokens.  The sliding window and the attention head are two different answers to the same question: how should nearby pixels inform each other?

<a id="extracting"></a>

## 3. Extracting representations

The model: **DINOv2** (small version, ~22M parameters, ~88 MB) — a vision foundation model from the Section 1 table this morning.  Its training is self-supervised. It sees no labels of any kind, just millions of images and an objective that amounts to *different views of the same image should get similar representations*.  Like ESM-2, whatever it learned, it learned from the data's own structure.

Loading follows the morning's pattern exactly — including the fallback if the model can't be fetched:

In [ ]:
import torch

shipped_embeddings = np.load('image_embeddings.npy')

try:
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
    model = AutoModel.from_pretrained('facebook/dinov2-small')
    model.eval()
    MODEL_AVAILABLE = True
    print('\nModel loaded:', sum(p.numel() for p in model.parameters()), 'parameters.')
except Exception as e:
    MODEL_AVAILABLE = False
    print('Model could not be loaded:', type(e).__name__)
    print('Falling back to the precomputed embeddings -- everything below still runs.')

One difference from the morning is worth seeing, and it needs one piece of background about how these models read an image.  A **vision transformer** reads an image the way the morning's models read a sequence: the image is cut into a grid of small square patches (14×14 pixels each, for DINOv2), and each patch is embedded as one token — an image becomes a "sentence" of patch-tokens.  The model's output is therefore one vector per patch... plus one extra.  Transformers prepend an artificial token at position 0, called the **CLS token** (short for "classification"), which carries no pixels of its own.  During training, the model learns to use that position as a scratchpad for a whole-image summary — attention lets it gather information from every patch.

That gives two reasonable ways to turn the output into one vector per image: average the patch vectors (mean-pooling, exactly as was done over residues this morning), or simply read off position 0.  For DINOv2 the CLS token is the standard choice, so that is what gets kept below.  Both conventions appear throughout the literature; for this dataset they perform almost identically, which is reassuring rather than disappointing.

In [ ]:
def extract(model, processor, images, batch_size=32):
    out = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            batch = processor(images=list(images[start:start+batch_size]), return_tensors='pt')
            hidden = model(**batch).last_hidden_state    # (b, 1 + patches, 384)
            out.append(hidden[:, 0].numpy())             # the CLS token
    return np.concatenate(out).astype(np.float32)

import time
if MODEL_AVAILABLE:
    t0 = time.time()
    embeddings = extract(model, processor, images)
    print(f'{len(images)} images embedded in {time.time()-t0:.1f} s on CPU')
    print('matches the precomputed file:', np.allclose(embeddings, shipped_embeddings, atol=1e-4))
else:
    embeddings = shipped_embeddings
    print('using precomputed embeddings:', embeddings.shape)

On my machine, about nine seconds for 600 images. Essentially the same cost as this morning's 450 proteins.  That number is the practical thesis of the day. On an ordinary laptop, foundation-model representations of a realistic scientific dataset cost seconds.

<a id="comparison"></a>

## 4. The comparison, third time

Now we take a third data type, and ask the same question we did with proteins. **Does the learned representation beat an honest simple baseline?**  For images, the natural simple baseline is the **color histogram**. For each image, just the distribution of pixel intensities in each channel, 24 numbers.  No model, no spatial information, only "how much of each color."  Since these cells are chemically stained, color is genuinely informative — this baseline is not a straw man.

One technical point before comparing, a call back to an earlier session.  Histogram features have tiny, varied scales, with a lot of variance in mean and variance across features. For logistic regression to work well, it needs them standardized. But standardizing with statistics computed on *all* the data before cross-validation is exactly the **leakage** pattern from yesterday.  The honest tool is a scikit-learn `Pipeline`. This bundles the scaler and the classifier. Cross-validation then re-fits the scaler *inside each fold*, on training rows only.  Watch how much this one line matters:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def color_hist(images, bins=8):
    out = []
    for im in images:
        h = [np.histogram(im[:, :, c], bins=bins, range=(0, 256))[0] for c in range(3)]
        v = np.concatenate(h).astype(float)
        out.append(v / v.sum())
    return np.array(out)

histograms = color_hist(images)
cv = KFold(5, shuffle=True, random_state=0)

plain = cross_val_score(LogisticRegression(max_iter=3000), histograms, labels, cv=cv).mean()
piped = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
                        histograms, labels, cv=cv).mean()
print(f'histogram features, no scaling:            {plain:.3f}')
print(f'histogram features, scaled inside folds:   {piped:.3f}')

From 0.39 to 0.83 — the single largest effect of feature scaling in this workshop, and the scaling was done *honestly*, inside the folds. This is why Wednesday's demo fit its scaler on the training part only, and why Thursday's leakage session called attention to how preprocessing can lead to faulty models.

Now the full comparison — three representations of the same 600 cells, same pipeline, same splits:

In [ ]:
probe_pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))

for name, X in [('color histogram (24 numbers)', histograms),
                ('raw pixels (12,288 numbers)', images.reshape(len(images), -1) / 255.0),
                ('DINOv2 embedding (384 numbers)', embeddings)]:
    acc = cross_val_score(probe_pipeline, X, labels, cv=cv).mean()
    print(f'{name:34s} CV accuracy = {acc:.3f}')
print(f'{"chance (8 classes)":34s} = 0.125')

Reading the table:

* **Raw pixels (0.61)** — 12,288 numbers, and the *worst* learned-from performance per number.  More features is not more information.
* **Color histogram (0.83)** — 24 numbers that genuinely work, because stain color tracks cell type.  Respect the baseline.
* **DINOv2 (0.93)** — the ~10 extra points over the histogram are *morphology*: nucleus shape, granularity, texture — the things a hematologist actually looks at, learned by a model that never saw a blood-cell label, and read out by the same linear probe as always.

Three data types, one pattern: words, proteins, and now images.  The claim from Section 1 of this morning — *whenever entities occur in measurable contexts, this method applies* — has now been demonstrated rather than asserted.

<a id="looking"></a>

## 5. Looking at the space

The interrogation tools from yesterday afternoon work unchanged — with one advantage unique to images. Nearest neighbors can be *looked at*.  For a query cell, here are its five nearest neighbors in embedding space:

In [ ]:
from scipy.spatial.distance import pdist, squareform

D = squareform(pdist(embeddings, 'cosine'))

fig = plt.figure(figsize=(14, 7), layout='tight')
axs = fig.subplots(3, 6)
rng = np.random.default_rng(4)
for row, q in enumerate(rng.choice(len(images), 3, replace=False)):
    order = [j for j in np.argsort(D[q]) if j != q][:5]
    axs[row, 0].imshow(images[q]); axs[row, 0].set_title(f'QUERY: {class_names[labels[q]]}', fontsize=10)
    for col, j in enumerate(order, start=1):
        axs[row, col].imshow(images[j])
        axs[row, col].set_title(class_names[labels[j]], fontsize=10)
    for ax in axs[row]: ax.axis('off')

Neighbors overwhelmingly share the query's class — and when they don't, looking at the images sometimes shows *why* the confusion is reasonable, which is precisely the kind of error analysis that numbers alone can't provide.

The global view, with PCA (and UMAP where installed):

In [ ]:
from sklearn.decomposition import PCA

coords = PCA(n_components=2).fit_transform(embeddings)
fig = plt.figure(figsize=(9, 6))
ax = fig.subplots(1)
for c in range(8):
    m = labels == c
    ax.scatter(coords[m, 0], coords[m, 1], s=10, label=class_names[c])
ax.legend(fontsize=9); ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2');

In [ ]:
%pip install -q umap-learn  # Colab does not preinstall umap-learn
try:
    import umap
    coords_u = umap.UMAP(n_components=2, random_state=0).fit_transform(embeddings)
    fig = plt.figure(figsize=(9, 6))
    ax = fig.subplots(1)
    for c in range(8):
        m = labels == c
        ax.scatter(coords_u[m, 0], coords_u[m, 1], s=10, label=class_names[c])
    ax.legend(fontsize=9); ax.set_title('UMAP projection')
except ImportError as e:
    print('umap could not be imported:', e)
    print('(pip install umap-learn, restart the kernel, and re-run -- optional.)')

<a id="homework"></a>

## Homework

Three functions, defined in the code cell below, mirroring this morning's set.  Edit the function definitions in the code cell below: replace each `raise RuntimeError` line with your own code, then re-run that cell before running the checks.  `embed_images` is for working on here in session.  `probe` is yesterday's probe with exactly one upgrade — the in-fold scaling `Pipeline` from Section 4 — and its docstring explains why that upgrade is the honest one.

In [ ]:
help(embed_images)
help(color_histogram)
help(probe)

**Check `embed_images`.**  It should reproduce the shipped `image_embeddings.npy` (tested on the first 32 images).  Expected: `reproduces the shipped file: True`

In [ ]:
if MODEL_AVAILABLE:
    mine = embed_images(model, processor, images[:32])
    print('shape:', mine.shape)
    print('reproduces the shipped file:', np.allclose(mine, shipped_embeddings[:32], atol=1e-4))
else:
    print('(model unavailable -- this check needs the live model; try it after the session)')

**Check `color_histogram`.**  Expected output:

```
shape: (600, 24)   rows sum to 1: True
```

In [ ]:
Xh = color_histogram(images)
print('shape:', Xh.shape, '  rows sum to 1:', np.allclose(Xh.sum(axis=1), 1))

**Check `probe`.**  Run on Section 4's representations, so it works before the other two functions are done.  Expected output:

```
histogram: 0.832   embeddings: 0.925
```

In [ ]:
ph = probe(histograms, labels)
pe = probe(embeddings, labels)
print(f'histogram: {ph:.3f}   embeddings: {pe:.3f}')

<a id="recipe"></a>

## The recipe, vision edition

For image data of any scientific kind — microscopy, histology, gels, camera traps, field photos:

1. **Get images into arrays** (any image loader ends at numpy).
2. **Find a vision model on the hub** — DINOv2 for general-purpose features; specialized models exist for pathology, cell painting, satellite imagery, and more.  Read the model card.
3. **Extract frozen embeddings on a CPU** (`embed_images` generalizes; mostly the processor call changes).
4. **Probe honestly** — with the in-fold scaling pipeline, grouped splits when images share a subject, slide, plate, or site (they usually do), and a simple baseline for humility.
5. **Look at the neighbors.**  Images are the one data type where the sanity check is literally visual — use that.

One caution carries into the final session: a model this good at reading images is equally good at reading *everything* in them — scanner signatures, staining batches, rulers and markings — and a probe cannot tell noble signal from shortcut.  What that means for believing results is where the workshop goes next.